# ChibiCreate — BENCHMARK COMPARATIVO: WAI-illustrious-SDXL

> ## BENCHMARK COMPARATIVO — NAO E PIPELINE OFICIAL
>
> Existe para responder **uma** pergunta:
>
> *"E possivel gerar nossa chibi diretamente com WAI-illustrious-SDXL e obter
> uma roupa mais fiel ao design original do que com FLUX.2 klein 4B?"*
>
> Nao altera o benchmark FLUX (`notebooks/flux2_klein_4b_eval.ipynb`), nem as
> Run 001/002/003 do FLUX, nem o Flow 01, nem os quality gates, nem o
> design transfer.

---

## LIMITACAO ARQUITETURAL — leia antes de executar

**SDXL nao tem multi-referencia nativa.** Isto nao e uma opcao que deixamos
de ligar; e uma diferenca de arquitetura:

| | mecanismo de referencia | referencias |
|---|---|---|
| FLUX.2 klein | `ReferenceLatent` (nativo) | 1, 2, 3... encadeaveis |
| Qwen-Image-Edit | `TextEncodeQwenImageEditPlus` | multiplos slots |
| **WAI / SDXL** | **nao tem** — so img2img | **1** |

O unico caminho com nodes Core e `VAEEncode -> KSampler(denoise<1.0)`, que
aceita **uma** imagem e preserva **composicao e cores**, nao **identidade**.

**Consequencia direta, declarada:** a RUN 003 deste notebook **nao e**
equivalente a RUN 003 do FLUX. Ela roda com **1 referencia**, nao 3. Chama-la
de "3-reference equivalent" seria mentira. IP-Adapter e ControlNet dariam
mais referencias, mas estao **fora de escopo** por instrucao.

O que fazemos: registrar exatamente quais referencias entraram, e comparar o
que e honestamente comparavel.


In [ ]:
#@title 0. Personagem e parametros do benchmark { display-mode: "form" }
#@markdown Unico ponto a mudar entre personagens. As celulas de execucao nao
#@markdown devem ser editadas.
CHARACTER_ID = "waifu_001"  #@param {type:"string"}
SEED = 42  #@param {type:"integer"}
#@markdown `denoise` e BASELINE_HYPOTHESIS, nao valor otimizado. Sem sweep.
DENOISE = 0.5  #@param {type:"slider", min:0.1, max:1.0, step:0.05}

MODEL_KEY = "wai_illustrious_sdxl"
WORKFLOW = "experimental/wai_illustrious_chibi"

print("personagem :", CHARACTER_ID)
print("modelo     :", MODEL_KEY)
print("workflow   :", WORKFLOW)
print("seed       :", SEED, "| denoise:", DENOISE)
print()
print("BENCHMARK COMPARATIVO. Nao e pipeline oficial.")
print("SDXL nao tem multi-referencia: a RUN 003 daqui usa 1 referencia.")


---

## Celula 1 — ambiente e repositorio

In [ ]:
#@title 1. Ambiente e repositorio { display-mode: "form" }
REPO_URL = "https://github.com/BloomRX/ChibiCreate"  #@param {type:"string"}
BRANCH   = "arena/01a07ece-chibicreate"  #@param {type:"string"}
#@markdown Mantenha marcado: garante que o runtime nao rode codigo antigo.
ATUALIZAR_REPO = True  #@param {type:"boolean"}

import os, sys, subprocess, pathlib

%cd /content
if not pathlib.Path("/content/ChibiCreate/.git").exists():
    !git clone --branch $BRANCH $REPO_URL ChibiCreate
elif ATUALIZAR_REPO:
    !cd /content/ChibiCreate && git fetch origin $BRANCH && git checkout -B $BRANCH origin/$BRANCH

%cd /content/ChibiCreate
!git log --oneline -1

for _m in [k for k in list(sys.modules) if k.startswith("chibi") or k.startswith("scripts.chibi")]:
    del sys.modules[_m]
sys.path.insert(0, "/content/ChibiCreate")
sys.path.insert(0, "/content/ChibiCreate/scripts")

!pip install -q pyyaml pillow numpy


---

## Celula 2 — preflight de hardware (portao)

Se o checkpoint nao couber, **para**. Sem fallback silencioso, sem CPU, sem
trocar de modelo, sem quantizar por conta propria.

In [ ]:
#@title 2. Preflight — GPU, VRAM, RAM, disco { display-mode: "form" }
import sys, json, shutil, subprocess

MIN_VRAM_GB = 10.0   # SDXL fp16 ~8-10 GB (registry, estimado)
MIN_DISK_GB = 12.0
MIN_RAM_GB  = 10.0

try:
    import torch
except ImportError:
    torch = None

print("=" * 62)
print("PREFLIGHT — detectado, nao presumido")
print("=" * 62)
print("Python :", sys.version.split()[0])
print("Torch  :", torch.__version__ if torch else "ausente")

if torch is None or not torch.cuda.is_available():
    print("CUDA   : INDISPONIVEL")
    raise SystemExit("BLOCKED — nenhuma GPU CUDA. Runtime -> GPU.")

props = torch.cuda.get_device_properties(0)
vram = props.total_memory / 1024 ** 3
free_disk = shutil.disk_usage("/content").free / 1024 ** 3
try:
    import psutil
    ram = psutil.virtual_memory().total / 1024 ** 3
except ImportError:
    ram = float(subprocess.check_output(
        ["awk", "/MemTotal/ {print $2/1048576}", "/proc/meminfo"]).strip())

GPU_INFO = {
    "name": props.name, "vram_total_gb": round(vram, 2),
    "vram_free_gb": round(torch.cuda.mem_get_info()[0] / 1024 ** 3, 2),
    "cuda": torch.version.cuda,
    "capability": f"{props.major}.{props.minor}",
    "torch": torch.__version__, "python": sys.version.split()[0],
    "bf16_supported": props.major >= 8,
    "ram_gb": round(ram, 2), "disk_free_gb": round(free_disk, 2),
}
for k, v in GPU_INFO.items():
    print(f"  {k:18} {v}")

print()
falhas = []
if vram < MIN_VRAM_GB:
    falhas.append(f"VRAM {vram:.1f} < {MIN_VRAM_GB} GB")
if free_disk < MIN_DISK_GB:
    falhas.append(f"disco {free_disk:.1f} < {MIN_DISK_GB} GB")
if ram < MIN_RAM_GB:
    falhas.append(f"RAM {ram:.1f} < {MIN_RAM_GB} GB")

if not GPU_INFO["bf16_supported"]:
    print("AVISO: sem bf16 nativo (capability < 8.0, ex. T4). Cai para fp16.")

if falhas:
    print("=" * 62); print("BLOCKED"); print("=" * 62)
    for f in falhas:
        print(" -", f)
    raise SystemExit("BLOCKED — nao fazer fallback silencioso.")

print("Hardware adequado.")
json.dump(GPU_INFO, open("/content/gpu_info_wai.json", "w"), indent=2)


---

## Celula 3 — entradas e hashes

As **mesmas** referencias do benchmark FLUX. Os arquivos originais nunca sao
modificados — so lidos e hasheados.

In [ ]:
#@title 3. Entradas — full_body, face, outfit { display-mode: "form" }
import hashlib, pathlib, json
import numpy as np
from PIL import Image

REF = pathlib.Path(f"/content/ChibiCreate/characters/{CHARACTER_ID}/reference")
ENTRADAS = {}
for papel, arq in [("full_body", "full_body.png"),
                   ("face", "face.png"),
                   ("outfit", "outfit.png")]:
    p = REF / arq
    if not p.exists():
        print(f"  [ausente] {papel}: {p}")
        continue
    b = p.read_bytes()
    im = Image.open(p)
    ENTRADAS[papel] = {
        "file": arq, "path": str(p),
        "artifact_sha256": hashlib.sha256(b).hexdigest(),
        "pixel_sha256": hashlib.sha256(
            np.array(im.convert("RGBA")).tobytes()).hexdigest(),
        "size": list(im.size), "bytes": len(b),
    }
    print(f"  {papel:10} {im.size}  {ENTRADAS[papel]['artifact_sha256'][:16]}")

assert "full_body" in ENTRADAS, "full_body.png e obrigatorio"
print()
print("Arquivos NAO sao modificados por este notebook — apenas lidos.")
json.dump(ENTRADAS, open("/content/wai_inputs.json", "w"), indent=2)


---

## Celula 4 — o modelo (registro e bloqueio de versao)

**Bloqueio real e declarado:** o Civitai esta fora da allowlist de egress da
sandbox onde este notebook foi escrito, entao `modelVersionId`, nome exato do
arquivo e SHA256 **nao puderam ser verificados** e estao `null` no registry.

Voce, no Colab, **enxerga** o Civitai. Preencha os campos abaixo com o que a
pagina realmente mostra. O notebook registra o que voce informar — e registra
tambem que veio de informe humano, nao de verificacao automatica.

In [ ]:
#@title 4. Fixar a versao do checkpoint { display-mode: "form" }
#@markdown Preencha com os dados REAIS da pagina do Civitai. Nao inventar.
CIVITAI_MODEL_ID = "827184"  #@param {type:"string"}
CIVITAI_VERSION_ID = ""  #@param {type:"string"}
CKPT_FILENAME = ""  #@param {type:"string"}
CKPT_SHA256_ESPERADO = ""  #@param {type:"string"}
LICENCA_EXIBIDA = "Commercial use allowed (conforme UI do Civitai)"  #@param {type:"string"}

from chibi import model_registry as mr

CFG = mr.get_model(MODEL_KEY)
print("label      :", CFG["label"])
print("pipeline   :", CFG["pipeline_type"])
print("refs suport:", CFG["references_supported"])
print("licenca    :", CFG["license_name"], "| verified:", CFG["license_verified"])
print("status     :", CFG["status"])
print()
print("LIMITACAO REGISTRADA:")
print(" ", CFG["reference_limitation"].strip())
print()

VERSAO = {
    "civitai_model_id": CIVITAI_MODEL_ID,
    "civitai_model_version_id": CIVITAI_VERSION_ID or None,
    "file": CKPT_FILENAME or None,
    "sha256_expected": CKPT_SHA256_ESPERADO or None,
    "license_displayed": LICENCA_EXIBIDA,
    "source_of_record": "informado por humano no Colab (Civitai bloqueado na sandbox)",
    "verified_by_agent": False,
}
if not CIVITAI_VERSION_ID or not CKPT_FILENAME:
    print("[HUMAN REVIEW REQUIRED] Versao NAO fixada.")
    print("Sem modelVersionId + nome do arquivo o benchmark nao e reproduzivel.")
    print("Preencha os campos acima antes de executar as runs.")
else:
    print("versao fixada:", CIVITAI_VERSION_ID, "|", CKPT_FILENAME)
json.dump(VERSAO, open("/content/wai_version.json", "w"), indent=2)


---

## Celula 5 — obter o checkpoint

O agente **nao** baixa modelo automaticamente (AGENTS.md). O download e uma
acao sua, explicita. Depois o SHA256 e conferido contra o que voce declarou
na celula 4 — se divergir, **para**.

In [ ]:
#@title 5. Download do checkpoint (acao explicita) { display-mode: "form" }
#@markdown Marque para baixar. Requer token do Civitai (a API exige auth).
BAIXAR_CHECKPOINT = False  #@param {type:"boolean"}
CIVITAI_TOKEN = ""  #@param {type:"string"}
#@markdown Alternativa: suba o .safetensors manualmente para
#@markdown `/content/ComfyUI/models/checkpoints/`.

import pathlib, hashlib
CKPT_DIR = pathlib.Path("/content/ComfyUI/models/checkpoints")
CKPT_DIR.mkdir(parents=True, exist_ok=True)

def sha256_of(p, chunk=1 << 22):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()

CKPT_PATH = None
if not BAIXAR_CHECKPOINT:
    achados = sorted(CKPT_DIR.glob("*.safetensors"))
    if achados:
        CKPT_PATH = achados[0]
        print("checkpoint ja presente:", CKPT_PATH.name)
    else:
        print("Download desativado e nenhum checkpoint encontrado.")
        print("Marque BAIXAR_CHECKPOINT ou suba o arquivo manualmente.")
elif not VERSAO["civitai_model_version_id"]:
    raise SystemExit("PARE: fixe a versao na celula 4 antes de baixar.")
else:
    url = ("https://civitai.com/api/download/models/"
           + VERSAO["civitai_model_version_id"])
    if CIVITAI_TOKEN:
        url += f"?token={CIVITAI_TOKEN}"
    destino = CKPT_DIR / (VERSAO["file"] or "wai_illustrious.safetensors")
    !wget -q --show-progress -O "$destino" "$url"
    CKPT_PATH = destino

if CKPT_PATH and CKPT_PATH.exists():
    real = sha256_of(CKPT_PATH)
    tam = CKPT_PATH.stat().st_size
    print(f"arquivo : {CKPT_PATH.name}")
    print(f"tamanho : {tam / 1e9:.2f} GB")
    print(f"sha256  : {real}")
    esperado = VERSAO.get("sha256_expected")
    if esperado:
        if real.lower() != esperado.lower():
            raise SystemExit("PARE: SHA256 diverge do declarado na celula 4.")
        print("confere com o declarado.")
    else:
        print("[HUMAN REVIEW REQUIRED] Sem SHA esperado — registre este valor.")
    VERSAO["sha256_actual"] = real
    VERSAO["size_bytes"] = tam


---

## Celula 6 — ComfyUI

In [ ]:
#@title 6. Subir o ComfyUI { display-mode: "form" }
import subprocess, time, urllib.request, json, pathlib, os

if not pathlib.Path("/content/ComfyUI").exists():
    !git clone -q https://github.com/comfyanonymous/ComfyUI.git /content/ComfyUI
    !pip install -q -r /content/ComfyUI/requirements.txt

COMFY_COMMIT = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd="/content/ComfyUI").decode().strip()
print("ComfyUI commit:", COMFY_COMMIT)
print("custom nodes  : NENHUM (workflow usa so nodes Core)")

LOG = open("/content/comfyui_wai.log", "w")
proc = subprocess.Popen(
    ["python", "main.py", "--listen", "127.0.0.1", "--port", "8188"],
    cwd="/content/ComfyUI", stdout=LOG, stderr=subprocess.STDOUT)

for i in range(120):
    time.sleep(5)
    try:
        with urllib.request.urlopen(
                "http://127.0.0.1:8188/system_stats", timeout=5) as r:
            stats = json.load(r)
        print(f"no ar apos ~{(i + 1) * 5}s")
        break
    except Exception:
        if proc.poll() is not None:
            print(open("/content/comfyui_wai.log").read()[-3000:])
            raise SystemExit("ComfyUI morreu ao iniciar")
else:
    raise SystemExit("ComfyUI nao respondeu em 10 min")

os.environ["CHIBI_COMFY_URL"] = "http://127.0.0.1:8188"


---

## Celula 7 — validar o workflow (portao)

Confere o grafo contra os nodes reais do servidor. **Se falhar, pare** — nao
edite o workflow para "fazer passar".

In [ ]:
#@title 7. Validar o grafo contra /object_info { display-mode: "form" }
%cd /content/ChibiCreate
!python -m scripts.chibi.cli comfy status --env cloud
print("=" * 62)
!python -m scripts.chibi.cli comfy validate --env cloud --workflow $WORKFLOW

import json
wf = json.load(open(f"workflows/{WORKFLOW}/v1.json"))
nodes = {k: v for k, v in wf.items() if not k.startswith("_")}
loads = [k for k, v in nodes.items() if v["class_type"] == "LoadImage"]
print()
print("LoadImage no grafo:", len(loads))
assert len(loads) == 1, "grafo mudou — a limitacao de 1 referencia foi alterada"
print("Confirmado: 1 referencia. Nao ha ReferenceLatent (SDXL nao tem).")
print("classes usadas:", sorted({v["class_type"] for v in nodes.values()}))


---

## RUN 001 — 1 referencia (`full_body`)

Equivalente conceitual a Run 001 do FLUX: uma referencia, parametros de
benchmark.

In [ ]:
#@title 8. RUN 001 { display-mode: "form" }
import yaml
_reg = yaml.safe_load(open("/content/ChibiCreate/config/model_eval_registry.yaml"))
PROMPT = _reg["base_prompt"].strip()
NEGATIVE = _reg["models"][MODEL_KEY]["negative_prompt_override"]

print("PROMPT:"); print(" ", PROMPT[:200], "...")
print("NEGATIVE:", NEGATIVE)
print()

%cd /content/ChibiCreate
!python -m scripts.chibi.cli experiment model-eval \
    --model wai-illustrious --character $CHARACTER_ID --seed $SEED \
    --prompt "$PROMPT"


## RUN 002 — repeticao EXATA da Run 001

Mesma seed, mesmo prompt, mesmo modelo, mesmo workflow, mesmos parametros,
mesma referencia. Testa reprodutibilidade. Hash diferente e resultado valido,
nao falha — e e exatamente o que queremos medir.

In [ ]:
#@title 9. RUN 002 — reprodutibilidade { display-mode: "form" }
%cd /content/ChibiCreate
!python -m scripts.chibi.cli experiment model-eval \
    --model wai-illustrious --character $CHARACTER_ID --seed $SEED \
    --prompt "$PROMPT"


## RUN 003 — NAO e equivalente a Run 003 do FLUX

A Run 003 do FLUX usa **3 referencias** (`full_body + face + outfit`) via
`ReferenceLatent`. **SDXL nao tem esse mecanismo.**

Esta run continua usando **1 referencia**. Ela existe para manter a numeracao
comparavel e para registrar a limitacao de forma explicita — **nao** para
fingir equivalencia.

A unica coisa que varia aqui e `denoise`, dentro do que o img2img realmente
oferece. Isso e adaptacao dentro das capacidades reais do workflow, e esta
declarado no recipe.

In [ ]:
#@title 10. RUN 003 — 1 referencia (limitacao declarada) { display-mode: "form" }
#@markdown `denoise` mais alto = mais liberdade para virar chibi, menos
#@markdown identidade preservada. Nao e sweep: e UMA variacao declarada.
DENOISE_RUN003 = 0.65  #@param {type:"slider", min:0.1, max:1.0, step:0.05}

print("=" * 62)
print("AVISO REGISTRADO NO RECIPE")
print("=" * 62)
print("Esta run usa 1 referencia (full_body), nao 3.")
print("SDXL nao possui multi-referencia nativa. face.png e outfit.png NAO")
print("entram no condicionamento. Nao chamar de '3-reference equivalent'.")
print("Referencias efetivamente usadas: ['full_body']")
print("Nao usadas:", [k for k in ENTRADAS if k != "full_body"])
print()

%cd /content/ChibiCreate
!python -m scripts.chibi.cli experiment model-eval \
    --model wai-illustrious --character $CHARACTER_ID --seed $SEED \
    --denoise $DENOISE_RUN003 --prompt "$PROMPT"


---

## Celula 11 — resultados e hashes

In [ ]:
#@title 11. Recipes, hashes e reprodutibilidade { display-mode: "form" }
import pathlib, json
from PIL import Image

base = pathlib.Path("experiments/model_eval")
runs = sorted([p for p in base.rglob("run_*") if (p / "recipe.json").exists()
               and "wai" in str(p).lower()])
print("execucoes WAI:", [r.name for r in runs])

RESUMO = []
for r in runs:
    rec = json.load(open(r / "recipe.json"))
    print()
    print("===", r.name, "=" * 40)
    for k in ("artifact_sha256", "output_sha256", "output_pixel_sha256",
              "seed", "execution_time", "reference_count"):
        if k in rec:
            print(f"  {k:22} {rec[k]}")
    print(f"  {'parameters':22} {rec.get('parameters')}")
    RESUMO.append({"run": r.name, "recipe": rec})
    img = r / "output.png"
    if img.exists():
        display(Image.open(img))

if len(runs) >= 2:
    a = json.load(open(runs[0] / "recipe.json"))
    b = json.load(open(runs[1] / "recipe.json"))
    print()
    print("REPRODUTIBILIDADE (run 001 vs run 002)")
    for campo in ("output_sha256", "output_pixel_sha256"):
        va, vb = a.get(campo), b.get(campo)
        igual = va == vb and va is not None
        print(f"  {campo:22} {'IDENTICO' if igual else 'DIFERENTE'}")
    print()
    print("Hash diferente NAO e falha: e resultado. Nao prometemos")
    print("determinismo absoluto entre execucoes.")


---

## Celula 12 — montagem comparativa

Sem ranking automatico. A leitura e humana.

In [ ]:
#@title 12. ORIGINAL vs FLUX vs WAI { display-mode: "form" }
import pathlib, json
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

def _abrir(p):
    p = pathlib.Path(p)
    return Image.open(p).convert("RGB") if p.exists() else None

ROOT = pathlib.Path("/content/ChibiCreate")
paineis = [(_abrir(ROOT / f"characters/{CHARACTER_ID}/reference/full_body.png"),
            "ORIGINAL (full_body)")]

_fx = sorted((ROOT / "experiments/model_eval").rglob("*flux*/run_*/output.png"))
if _fx:
    paineis.append((_abrir(_fx[0]), "FLUX RUN 001 (1 ref)"))

flux003 = ROOT / f"characters/{CHARACTER_ID}/chibi/run_003_output.png"
if flux003.exists():
    paineis.append((_abrir(flux003), "FLUX RUN 003 (3 refs)"))

_wai = sorted([p for p in (ROOT / "experiments/model_eval").rglob("run_*/output.png")
               if "wai" in str(p).lower()])
for i, p in enumerate(_wai[:3], start=1):
    rotulo = f"WAI RUN 00{i}" + (" (1 ref)" if i != 3 else " (1 ref — NAO 3)")
    paineis.append((_abrir(p), rotulo))

paineis = [(im, t) for im, t in paineis if im is not None]
n = len(paineis)
fig, ax = plt.subplots(1, n, figsize=(5 * n, 7))
for a, (im, t) in zip(np.atleast_1d(ax), paineis):
    a.imshow(im); a.set_title(t, fontsize=10); a.axis("off")
plt.tight_layout(); plt.show()

print("=" * 62)
print("[HUMAN REVIEW REQUIRED] — nao ha ranking automatico")
print("=" * 62)
print("Eixos: STYLE / IDENTITY / DESIGN_PRESERVATION")
print("Eixo PRINCIPAL: DESIGN_PRESERVATION")
print()
print("Avalie item a item:")
for item in ["roupa", "capa", "ornamentos dourados", "acessorios", "cabelo",
             "chifres", "proporcao corporal", "silhueta", "cores"]:
    print("  [ ]", item)
print()
print("Pergunta do benchmark:")
print('  "O WAI produz uma roupa MAIS FIEL ao design original que o FLUX?"')
print()
print("  SIM -> WAI vira candidato a ROTA DIRETA")
print("  NAO -> FLUX segue como baseline")
print()
print("Lembre da assimetria: WAI rodou com 1 referencia, FLUX run_003 com 3.")


In [ ]:
#@title 13. Metadata da sessao { display-mode: "form" }
import json, pathlib

meta = {
    "benchmark": "wai_illustrious_sdxl_vs_flux2_klein_4b",
    "purpose": "comparative_benchmark_only",
    "is_official_pipeline": False,
    "character": CHARACTER_ID,
    "infrastructure": "google_colab",
    "infrastructure_status": "EXPERIMENTAL_TEMPORARY",
    "gpu": json.load(open("/content/gpu_info_wai.json")),
    "model": {"key": MODEL_KEY, **VERSAO},
    "inputs": ENTRADAS,
    "workflow": WORKFLOW,
    "comfyui_commit": COMFY_COMMIT,
    "custom_nodes": [],
    "custom_nodes_note": "Nenhum. Somente nodes Core do ComfyUI.",
    "prompt": PROMPT,
    "negative_prompt": NEGATIVE,
    "seed": SEED,
    "multi_reference_supported": False,
    "multi_reference_note": (
        "SDXL nao possui mecanismo nativo de multi-referencia. A RUN 003 "
        "deste benchmark usa 1 referencia, ao contrario da RUN 003 do FLUX "
        "que usa 3. NAO sao equivalentes."),
    "references_used": ["full_body"],
    "references_not_used": [k for k in ENTRADAS if k != "full_body"],
    "license_note": (
        "FAIPL-1.0-SD nao verificada em fonte primaria (Civitai fora da "
        "allowlist da sandbox). commercial_status: pending_human_review."),
    "cost": 0,
    "cost_note": "No direct GPU cost observed in this session.",
    "limitations": [
        "Sessao efemera: /content e perdido ao desconectar.",
        "GPU nao garantida entre sessoes.",
        "Colab NAO e infraestrutura permanente do projeto.",
        "Sem determinismo garantido entre execucoes.",
    ],
}
p = pathlib.Path("/content/ChibiCreate/experiments/model_eval/wai_session.json")
p.parent.mkdir(parents=True, exist_ok=True)
p.write_text(json.dumps(meta, indent=2, default=str))
print(json.dumps(meta, indent=2, default=str)[:2500])


---

## PARE AQUI

Fim do escopo deste notebook.

**Nao** implementar a rota FLUX -> WAI ainda. Primeiro medimos
`ORIGINAL -> WAI` e comparamos com `ORIGINAL -> FLUX`.

So **se** o WAI mostrar vantagem real em **DESIGN_PRESERVATION** e que
partimos para `FLUX RUN 003 -> WAI`.

### [HUMAN REVIEW REQUIRED]

- Fixar a versao do checkpoint (`modelVersionId`, arquivo, SHA256).
- Ler a licenca da versao especifica na pagina do Civitai.
- Avaliar os tres eixos e concluir com **uma** marca:
  `PROMISING` / `INSUFFICIENT` / `BLOCKED`.

Nada de LoRA, ControlNet, IP-Adapter, inpainting, design transfer, master,
animacao ou pipeline oficial.
